# Running the NBSOS

This notebook will demonstrate how a NBSOS scan is setup and run. To get started, import the Scan class, that implements the NBSOS, and make some further imports needed to define context, NbS, and benefits of interest.

In [ ]:
# nbsos-specific imports
# this assumes that urban_v1 is installed as a local package (pip install -e .) in src dir
# requirements.txt in the urban_v1 folder should list all required packages. 
from nbsos.urban_v1 import Scan, ScanContext, DirectDownloadHandler, GcsHandler, NbsTypes, Benefits, ScanBenefit

# other imports
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## General approach

To run the scan requires the following methodological steps. The Scan class allows you to walk through each of these steps.

* Retrieve and prepare flood hazard data (this is only needed if flood benefits are of interest);
* Set environment variables; 
* Define NBS of interest;
* Instantiate a scan context;
* Instantiate a new scan;
* Run scan preparation and download all other data;
* Run suitability model
* Run cluster exploration and identify locally appropriate threshold through visual inspection for determining protection and creation opportunities;
* Run clustering to create the NBS protection mask and the NBS creation masks;
* Define the benefits of interest;
* Run creation benefit models;
* Run protection benefit models;
* Run priority area (and exposure) mapping;
* Run multicriteria analysis;
* If building solutions are relevant for the case, run building solutions method.

## Set environment variables


Setting up a scan requires setting a couple of environment variables, as shown below. Moreover, to assess pluvial flood benefits, the scan requires flood extent/flood depth data as flood hazard data (provided by Fathom, or e.g., available from JBA). Defended pluvial flood data is used for assessing pluvial flood benefits, but data may also be pre-processed, e.g., to combine fluvial (defended) and pluvial flood data, as required for a case. Retrieved/pre-processed data will be ingested from a defined location, below referred to as *flood_data_path*. Moreover, the return periods for the data needs to be specified, using a Dict[str,int] (example below shows the naming convention for Fathom, but this can obviously vary):

In [ ]:
# first, define these common environment parameters
local_root = 'path\\to\\local_root'
country_name = "name"
city_name = "name"
aoi_path = 'path\\to\\wgs84\\shapefile.shp' 
location_is_in_africa = False # set as required; this determines which data is retreived from Google Earth Engine, depending on location

# the flood hazard data path should be set to the location where flood raster data is stored per return period
# this is the data you downloaded or pre-processed earlier
# if flood benefits are not of interest, set this variable to None
flood_data_path = 'path\\to\\data' # or: None

# the flood mapping associates each flood raster file with its corresponding return period; in case of Fathom data, no change is needed. 
# Otehrwise, set the filenames in the dictionary's keys and the return periods in the values according to your data files.
# You can leave this as-is also in the case that flood benefits are not of relevance; it would be ignored in that case.
flood_mapping = {
    'P_1in5.tif' : 5,
    'P_1in10.tif' : 10,
    'P_1in20.tif' : 20,
    'P_1in50.tif' : 50,
    'P_1in100.tif' : 100,
    'P_1in200.tif' : 200,
    'P_1in500.tif' : 500,
    'P_1in1000.tif' : 1000
}

## Define NbS families

For each scan, the NbS families/types of NbS that are of interest need to be defined. Supported NbS families are defined in the NbsTypes enumeration. NbS of interest are provided to the Scan class in the form of a list.

In [ ]:
# define NbS of interest
# comment NBS families not relevant for a given case
nbs_list = [
    NbsTypes.BioretentionAreas,
    NbsTypes.UrbanForest,
    NbsTypes.GreenCorridors,
    NbsTypes.InlandWetlands,
    NbsTypes.OpenGreenSpace,
    NbsTypes.RiverRenaturation,
    NbsTypes.TerracesAndSlopes
]

## Instantiate a scan context

The ScanContext class is used as container to define the local path to the NBSOS data repository, and further parameters relevant for a scan. These include, e.g., name of country and name of city of the case to be assessed. Further environment variables may be defined through the scan context, as needed. Define the parameters below. ***Note that the AOI shapefile is expected in WGS84 CRS.***

In [ ]:
# instantiate ScanContext class. Typically, no changes needed here as this just references the environment variables set above.
scan_context: ScanContext = ScanContext()
scan_context.local_root = local_root
scan_context.country = country_name
scan_context.city = city_name
scan_context.local_area_of_interest_file = aoi_path
scan_context.location_in_africa = location_is_in_africa

## Instantiate the NBSOS Scan class 

To run a scan, the Scan class needs to be instantiated, passing the scan context and the list of NbS of interest.

In [ ]:
# instantiate Scan class. Typically, no changes needed here as everything was set above.
my_scan = Scan(
    scan_context=scan_context,
    nbs_list=nbs_list
)

# Instantiate a storage handler

In the next step, a StorageHandler needs to be instantiated and passed to the scan. The storage handler ingests local data (i.e., the flood hazard data), takes care of processing and downloading data from GoogleEarthEngine or OpenStreetMap, and prepares several indicators locally. At this time, two storage handlers are available; use either 

* GcsStorageHandler: Uses Google Cloud Storage (GCS) as intermediate storage provider, and downloads data first to GCS and then to local device. Supports upload of data to GCS for archiving. This is internally the default. Requires name of bucket and path to credentials; or

* DirectDownloadHandler: Implements a direct download mechanism to retrieve data from GEE directly to disk. This is faster than using GCS, but may fail for large rasters/larger AOI. Requires name of GEE-connected cloud project.

*Note that flood data path can be None if no pluvial flood benefits are to be assessed. This should have been set above.* 

In [ ]:
# instantiate DirectDownloadHandler for downloading data directly from GEE
my_storage_handler = DirectDownloadHandler(
    scan_context=scan_context,
    flood_hazard_data_folder=flood_data_path,
    file_to_probability_mapping=flood_mapping
)
# set name of project used to process data in GEE
project_name = 'name of project in GEE'
my_storage_handler.project_name = project_name


... or use ...

In [ ]:
# instantiate GcsHandler for downloading data via Google Cloud Storage
my_storage_handler = GcsHandler(
    scan_context=scan_context,
    flood_hazard_data_folder=flood_data_path,
    file_to_probability_mapping=flood_mapping
)
my_storage_handler.credentials = 'path to credentials json'
my_storage_handler.bucket_name = 'urban-nbs-scan-bucket'

Once either of the two is defined, the instantiated storage handler should be passed to the scan

In [ ]:
# pass storage handler to scan. This will also intialize the handler
my_scan.set_storage_options(storage_handler=my_storage_handler)

## Define benefits

Benefits to be assessed are passed as a list of ScanBenefit objects. ScanBenefit encapsulate the type of benefit to be assessed, and allow to specify parameters for assessment as Dict[str|any] for the params parameter. Supported keywords for the params dictionary correspond to the parameters of methods called for determining creation benefits in the Scan class:

* get_biodiversity_creation_benefits
* get_heat_creation_benefits
* get_recreation_creation_benefits (for recreation and social cohesion)
* get_pluvial_flood_creation_benefits
* get_job_creation_creation_benefits
* get_erosion_creation_benefits

Note that for the PluvialFlooding benefit, return_periods always need to be specified.

In [ ]:
# define benefits of interest
# define additional parameters as needed
benefits = [
    ScanBenefit(Benefits.UrbanHeat),
    ScanBenefit(Benefits.Recreation),
    ScanBenefit(Benefits.SocialCohesion),
    ScanBenefit(Benefits.PluvialFlooding, params={'return_periods' : list(flood_mapping.values())}),
    ScanBenefit(Benefits.Biodiversity),
    ScanBenefit(Benefits.AvoidedErosion)
]

# Run analytics

## Prepare the scan data

The run_prepartion method will take care of processing input data in GEE, downloading this data to the local device, and some client-side processing of data. By default, all parameters of this method are set to True so that:

* GEE is run: This processes indicators in GEE, and in case of the direct download handler, downloads the processed data to the local device.
* Download is run: In the case of the direct download handler, this downloads several additional (non-GEE) data, e.g., from OpenStreetMap.  
* Client-side processing is run: This performs some processing of data on the local device, including the preparation of flood hazard data for the scan. 

Setting any of these parameters to False allows to conduct only specific steps, but this should normally not be needed. 

In [ ]:
# prepare indicators and download to local device
my_scan.run_preparation(
   run_gee=True, 
   run_download=True, 
   run_client_computations=True
)

## Run the suitability model

Having run the run_preparation method, all required data should now be available. In the next step, the suitability model needs to be run. This identifies areas suitable for NbS families based on suitability rulesets. The floodplain_path parameter can be ignored or set to None if no RiverFloodplain NBS family is included. 



In [ ]:
# run suitability model
my_scan.run_suitability() 

## Identify protection and creation opportunities  

In the next step, protection and creation opportunities need to be identified. The former refers to existing green elements, the latter to areas that provide opportunities for the implementation of NbS families. Protection and creation opportunities are identified using an explorative clustering, and a visual inspection of clusters to determine a clustering threshold value that separates both types most effectively.

The explorative clustering is conducted using the explore_clusters method. The result of the operation is stored as raster file 'clusters.tif' in the 'cluster_maps' folder and needs to be inspected visually by the user. A threshold needs to be chosen as such that it is the first cluster value that, based on user judgement, does not overlap with vegetation considered healthy/not degraded.

The most-appropriate threshold value chosen should be noted by the user. It is subsequently required by the scan.

In [ ]:
# run clustering for visual inspection
my_scan.explore_clusters() 

The identified clustering threshold is noted below, and will be used to determine protection and creation opportunities. For creation opportunities, an optional masking mechanism is implemented, as follows:

* Currently, only shapefiles are supported as masks. 
* All masks should be located in the “masks” folder within the main folder of the scan. 
* Masking will be conducted on the basis of geometries contained in the shapefiles, not based on any attribute. Rasterized masks are saved to the masks folder for reference.
* Creation opportunities will be overwritten with the masks applied. No creation opportunities should appear in the specified areas.

This allows to mask creation opportunities in undesired areas, e.g., airports.

In [ ]:
# the cluster threshold derived from visual inspection to be inserted here, for example 8.
cluster_threshold = 8

# define masks for creation opportunities, as needed,
# as a list of str referring to shapefiles in the masks folder ['shapefile1_name.shp', '...']
# set to None if no masks are needed (None is the default value)
masks = None

my_scan.run_clustering(
    clustering_threshold=cluster_threshold,
    exclusion_masks=masks
)

## Run benefits models

Once a suitable threshold for the identification of protection and creation opportunities has been selected and protection and creation opportunities have been identified, the benefit models can be run. 

* Creation benefits are determined by the run_creation_benefits method. The results are stored at various locations depending on the module, but are ingested by the run_multicriteria_analysis method that computes outputs relevant for mapping.

* Protection benefits are determined by the run_protection_benefits method. Note that estimating protection benefits requires certain data that is produced as part of the creation models, therefore, protection benefits should always be run after running creation benefit estimation. Results are written to the output/protection folder.

In [ ]:
# run creation benefit models
# see method in the nbsos.py
my_scan.run_creation_benefits(
    benefit_parameters=benefits
)

For protection benefits, two masks designating protection opportunities are available: (i) a mask identifying protection opportunities as intersect of creation opportunities in non-degraded vegetation (denoted by suitability); and (ii) a mask that aligned to the clustering threshold used, with all pixels > the chosen clustering threshold designated protection opportunities (denoted by lppi). 

The suitability-type mask is a legacy option, the lppi-type mask should be used as default (and is set as default if not specified by the user).

Note that everywhere, lppi is used as default value for protection opportunities.

In [ ]:
# by default, use lppi protection mask type
my_scan.run_protection_benefits(
   benefit_parameters=benefits
)

## Determine priority areas

Identify areas to prioritize NbS. Outputs are written to the output/priority folder.

In [ ]:
# create priority areas
my_scan.run_priority(
    benefit_parameters=benefits
)

## Multicriteria analysis

Finally, the NBSOS provides a multicriteria analysis through the run_multicriteria_analysis, that identifies the maximum benefit across NbS families, and that also identifies the optimal NbS, i.e., the NbS family achieving this maximum benefit. This method will iterate over various weighting schemas, where weights correspond to the relative importance of a given benefit in determining the maximum impact. The weighting schemas evaluated include the following, which can be run separately, as desired:

* equal weighting, i.e., all benefits are of equal importance;
* benefit-heavy schemas, i.e., each of the assessed benefits is treated as focal benefit receiving a comparatively high weight (the focal weight), whilst other benefits receive an equal weights indicating lower importance overall; and 
* benefit-only schemas, that maximize the importance of each focal benefit, whereas impacts of NbS regarding the remaining benefits are not considered.   
* custom weighting schemas defind as custom_scenarios dict, where the dictionary key is the name of a scenario and is used in file naming conventions.   

This method will also create binned rasters of relative performance of NBS families per each benefit, for mapping.

Results are written as follows:

* Relative binned performance of NBS families: output/nbs_compare folder, one subfolder per benefit
* Weighting schemes: output/mca, where folders for each schema are located, and results per NBS family and benefit are located in these folders 

In [ ]:
# run multicriteria analysis, including comparion of relative performances of nbs per benefit, etc.
# adapt focal weight for benefit heavy-scenarios, as needed
focal_weight = 0.8

# specify whether to run scenarios (equal, heavy, only)
assess_equal_weights_scenario = True
assess_benefit_heavy_scenarios = True
assess_benefit_only_scenarios = True

# define custom scenarios, as needed. 
# This is a dict of dicts (key refers to scenario name, and value to a dict of benefit-weight pairs).
# If not needed, set to None.
custom_scenarios = {
    'balanced' : {
        Benefits.PluvialFlooding : 0.3,
        Benefits.AvoidedErosion : 0.5,
        Benefits.UrbanHeat : 0.1,
        Benefits.Biodiversity : 0.0,
        Benefits.Recreation : 0.0,
        Benefits.JobsCreation : 0.1,
    }
}

# in addition, specify percentile thresholds to define the highest X% benefits to be exported
# E.g., [80, 90, 95] for highest 20%, 10%, and 5% benefits, respectively.
# By default, 80th percentile, i.e., highest 20% benefits. 
highest_benefits_percentiles = [80, 90]

# run multicriteria analysis
my_scan.run_multicriteria_analysis(
   benefit_parameters=benefits,
   heavy_focal_weight=focal_weight,
   assess_equal_weights_scenario=assess_equal_weights_scenario,
   assess_benefit_heavy_scenario=assess_benefit_heavy_scenarios,
   assess_benefit_only_scenario=assess_benefit_only_scenarios,
   custom_scenarios=custom_scenarios, # or set to None 
   highest_benefits_percentiles=highest_benefits_percentiles
)

## Building solutions

Building solutions as a NBS family can be assessed using the run_building_solutions method, but they are not included in NbsTypes in a meaningful way. The building solutions model is separate from other benefit models, is only specified for heat and flood benefits, and requires protection benefits and priority areas to be estimated. The module logic is as follows:

* Identify areas where no protection or creation opportunities are present;
* and where building cover is high;
* and where priority for heat or flood is high.

A building score is estimated for heat and flooding based on these criteria. The higher the building score, the higher the building priority score for the respective benefit. Areas with no building scores may be masked by setting the exclude_non_built_areas to True.

In [ ]:
# run building solutions, after clustering and priority area identification
my_scan.run_building_solutions(
    benefit_parameters=benefits, 
    exclude_non_built_areas=False
)

## Close the scan

Finally, upon completion, data can be archived to GCS and scan can be closed (this is only defined for the GCS Storage Handler).

In [ ]:
# close scan, save logs
my_scan.archive_data()
my_scan.close()
